##### 自动生成包含 7 个工作表、条件格式、公式计算、数据验证和图表的专业投资组合跟踪 Excel 文件。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
投资组合跟踪模板自动生成器
功能：生成包含 7 个工作表的 Excel 文件，含条件格式、公式、数据验证、图表
版本：v1.0
日期：2026-03-20
"""

import pandas as pd
import numpy as np
from datetime import datetime
import xlsxwriter
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, Color
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import PieChart, Reference
from openpyxl.utils import get_column_letter

# ==================== 配置参数 ====================

# 18 只精选标的核心数据
STOCKS_DATA = [
    {'代码': '600938', '名称': '中国海油', '板块': '上游油气', '目标仓位': 10, '当前价': 42.24, '入场下限': 38.5, '入场上限': 40.5, '止损价': 36.0, '目标价': 52.0, '仓位上限': 12, '股息率': 5.2, 'PE': 14.4},
    {'代码': '601857', '名称': '中国石油', '板块': '上游油气', '目标仓位': 8, '当前价': 12.50, '入场下限': 11.8, '入场上限': 12.8, '止损价': 10.8, '目标价': 16.0, '仓位上限': 10, '股息率': 4.3, 'PE': 13.2},
    {'代码': '600256', '名称': '广汇能源', '板块': 'LNG', '目标仓位': 6, '当前价': 7.00, '入场下限': 6.2, '入场上限': 7.2, '止损价': 5.5, '目标价': 10.0, '仓位上限': 8, '股息率': 3.5, 'PE': 33.1},
    {'代码': '600803', '名称': '新奥股份', '板块': 'LNG', '目标仓位': 6, '当前价': 22.00, '入场下限': 19.5, '入场上限': 22.5, '止损价': 18.0, '目标价': 28.0, '仓位上限': 8, '股息率': 3.0, 'PE': 13.7},
    {'代码': '601808', '名称': '中海油服', '板块': '油服', '目标仓位': 6, '当前价': 17.40, '入场下限': 16.5, '入场上限': 18.0, '止损价': 15.0, '目标价': 23.0, '仓位上限': 8, '股息率': 2.1, 'PE': 19.4},
    {'代码': '002353', '名称': '杰瑞股份', '板块': '油服', '目标仓位': 6, '当前价': 99.00, '入场下限': 90.0, '入场上限': 100.0, '止损价': 85.0, '目标价': 125.0, '仓位上限': 10, '股息率': 0.9, 'PE': 36.1},
    {'代码': '601088', '名称': '中国神华', '板块': '煤炭化工', '目标仓位': 8, '当前价': 46.00, '入场下限': 43.0, '入场上限': 47.0, '止损价': 40.0, '目标价': 55.0, '仓位上限': 10, '股息率': 7.0, 'PE': 13.0},
    {'代码': '600426', '名称': '华鲁恒升', '板块': '煤炭化工', '目标仓位': 4, '当前价': 35.60, '入场下限': 36.0, '入场上限': 41.0, '止损价': 33.0, '目标价': 48.0, '仓位上限': 8, '股息率': 2.5, 'PE': 20.0},
    {'代码': '600406', '名称': '国电南瑞', '板块': '特高压', '目标仓位': 12, '当前价': 30.70, '入场下限': 29.0, '入场上限': 31.5, '止损价': 27.0, '目标价': 40.0, '仓位上限': 15, '股息率': 2.0, 'PE': 25.0},
    {'代码': '000400', '名称': '许继电气', '板块': '特高压', '目标仓位': 6, '当前价': 33.60, '入场下限': 31.0, '入场上限': 35.0, '止损价': 28.0, '目标价': 43.0, '仓位上限': 8, '股息率': 1.5, 'PE': 30.0},
    {'代码': '300750', '名称': '宁德时代', '板块': '新能源', '目标仓位': 12, '当前价': 400.50, '入场下限': 380.0, '入场上限': 410.0, '止损价': 360.0, '目标价': 500.0, '仓位上限': 12, '股息率': 1.0, 'PE': 28.0},
    {'代码': '300274', '名称': '阳光电源', '板块': '新能源', '目标仓位': 8, '当前价': 165.10, '入场下限': 155.0, '入场上限': 170.0, '止损价': 145.0, '目标价': 210.0, '仓位上限': 12, '股息率': 1.2, 'PE': 22.0},
    {'代码': '601899', '名称': '紫金矿业', '板块': '黄金', '目标仓位': 10, '当前价': 32.40, '入场下限': 30.0, '入场上限': 34.0, '止损价': 28.0, '目标价': 43.0, '仓位上限': 12, '股息率': 2.5, 'PE': 17.3},
    {'代码': '600489', '名称': '中金黄金', '板块': '黄金', '目标仓位': 8, '当前价': 26.70, '入场下限': 24.5, '入场上限': 27.5, '止损价': 22.5, '目标价': 34.0, '仓位上限': 10, '股息率': 3.8, 'PE': 30.0},
    {'代码': '600760', '名称': '中航沈飞', '板块': '军工', '目标仓位': 8, '当前价': 50.87, '入场下限': 48.0, '入场上限': 50.5, '止损价': 45.0, '目标价': 65.0, '仓位上限': 10, '股息率': 1.2, 'PE': 45.0},
    {'代码': '002179', '名称': '中航光电', '板块': '军工', '目标仓位': 7, '当前价': 34.80, '入场下限': 33.0, '入场上限': 35.5, '止损价': 30.0, '目标价': 45.0, '仓位上限': 8, '股息率': 2.2, 'PE': 29.0},
    {'代码': '603019', '名称': '中科曙光', '板块': '政策方向', '目标仓位': 8, '当前价': 84.80, '入场下限': 80.0, '入场上限': 85.0, '止损价': 75.0, '目标价': 110.0, '仓位上限': 10, '股息率': 0.8, 'PE': 60.0},
    {'代码': '600536', '名称': '中国软件', '板块': '政策方向', '目标仓位': 4, '当前价': 45.00, '入场下限': 42.0, '入场上限': 48.0, '止损价': 38.0, '目标价': 60.0, '仓位上限': 8, '股息率': 0.5, 'PE': 80.0},
]

# 全局参数
INITIAL_CAPITAL = 100000  # 初始本金 10 万
RISK_FREE_RATE = 0.025  # 无风险利率 2.5%
COMMISSION_RATE = 0.00025  # 佣金 0.025%
STAMP_TAX = 0.001  # 印花税 0.1%
REBALANCE_THRESHOLD = 0.15  # 再平衡阈值 15%
STOP_LOSS_THRESHOLD = -0.15  # 止损阈值 -15%

# 文件名
FILENAME = f'投资组合跟踪模板_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'


# ==================== 样式定义 ====================

class Styles:
    """定义 Excel 样式"""
    
    # 字体
    header_font = Font(name='微软雅黑', size=11, bold=True, color='FFFFFF')
    normal_font = Font(name='微软雅黑', size=10)
    warning_font = Font(name='微软雅黑', size=10, bold=True, color='FF0000')
    
    # 填充色
    header_fill = PatternFill(start_color='1F4E79', end_color='1F4E79', fill_type='solid')  # 深蓝
    green_fill = PatternFill(start_color='C6EFCE', end_color='C6EFCE', fill_type='solid')  # 浅绿
    red_fill = PatternFill(start_color='FFC7CE', end_color='FFC7CE', fill_type='solid')  # 浅红
    yellow_fill = PatternFill(start_color='FFEB9C', end_color='FFEB9C', fill_type='solid')  # 浅黄
    blue_fill = PatternFill(start_color='D6EAF8', end_color='D6EAF8', fill_type='solid')  # 浅蓝
    orange_fill = PatternFill(start_color='FDEBD0', end_color='FDEBD0', fill_type='solid')  # 浅橙
    gray_fill = PatternFill(start_color='F2F2F2', end_color='F2F2F2', fill_type='solid')  # 浅灰
    
    # 对齐
    center_align = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left_align = Alignment(horizontal='left', vertical='center')
    right_align = Alignment(horizontal='right', vertical='center')
    
    # 边框
    thin_border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )


# ==================== 工作表创建函数 ====================

def create_parameters_sheet(wb):
    """创建工作表 7：参数设置"""
    ws = wb.create_sheet('07_参数设置', 0)  # 放在第一个位置
    
    # 全局参数
    global_params = [
        ['全局参数设置', '', '', ''],
        ['参数名称', '当前值', '说明', ''],
        ['初始本金', INITIAL_CAPITAL, '组合起始资金（元）', ''],
        ['无风险利率', RISK_FREE_RATE, '用于夏普比率计算（年化 2.5%）', ''],
        ['交易佣金率', COMMISSION_RATE, '买入佣金 0.025%', ''],
        ['印花税率', STAMP_TAX, '卖出印花税 0.1%', ''],
        ['再平衡阈值', REBALANCE_THRESHOLD, '权重偏离超过 15% 触发再平衡', ''],
        ['止损阈值', STOP_LOSS_THRESHOLD, '单标的亏损 15% 触发止损检查', ''],
        ['数据更新日', datetime.now().strftime('%Y-%m-%d'), '最后更新时间', ''],
        ['', '', '', ''],
        ['18 只标的策略参数表', '', '', ''],
    ]
    
    # 表头
    header = ['代码', '名称', '板块', '目标仓位%', '入场下限', '入场上限', '止损价', '目标价', '仓位上限%', '股息率%', 'PE(TTM)', '检视频率']
    
    for row_idx, row_data in enumerate(global_params, 1):
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            if row_idx <= 2 or row_idx == 11:
                cell.font = Styles.header_font
                cell.fill = Styles.header_fill
                cell.alignment = Styles.center_align
            else:
                cell.font = Styles.normal_font
                cell.alignment = Styles.left_align
    
    # 写入表头
    for col_idx, col_name in enumerate(header, 1):
        cell = ws.cell(row=12, column=col_idx, value=col_name)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 写入标的数据
    for row_idx, stock in enumerate(STOCKS_DATA, 13):
        data = [
            stock['代码'], stock['名称'], stock['板块'], stock['目标仓位'],
            stock['入场下限'], stock['入场上限'], stock['止损价'], stock['目标价'],
            stock['仓位上限'], stock['股息率'], stock['PE'], '月度'
        ]
        for col_idx, value in enumerate(data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.alignment = Styles.center_align
            cell.border = Styles.thin_border
            # 数值列右对齐
            if col_idx in [4, 5, 6, 7, 8, 9, 10, 11]:
                cell.alignment = Styles.right_align
    
    # 调整列宽
    column_widths = [12, 15, 12, 12, 12, 12, 12, 12, 12, 12, 12, 10]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    print("✅ 工作表 7：参数设置 - 创建完成")


def create_positions_sheet(wb):
    """创建工作表 2：持仓明细"""
    ws = wb.create_sheet('02_持仓明细')
    
    # 表头
    headers = [
        '股票代码', '股票名称', '所属板块', '目标仓位%', '持仓数量', '持仓成本',
        '当前股价', '持仓市值', '实际仓位%', '浮动盈亏', '盈亏比例%',
        '入场下限', '入场上限', '止损价', '目标价', '股息率%', 'PE(TTM)',
        '技术信号', '基本面评级', '操作建议', '备注'
    ]
    
    # 写入表头
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 写入标的数据
    for row_idx, stock in enumerate(STOCKS_DATA, 2):
        data = [
            stock['代码'], stock['名称'], stock['板块'], stock['目标仓位'],
            0, 0, stock['当前价'], 0, 0, 0, 0,
            stock['入场下限'], stock['入场上限'], stock['止损价'], stock['目标价'],
            stock['股息率'], stock['PE'], '观望', '良好', '持有', ''
        ]
        for col_idx, value in enumerate(data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.alignment = Styles.center_align
            cell.border = Styles.thin_border
        
        # 盈亏列条件格式（后续用 openpyxl 条件格式）
        ws.cell(row=row_idx, column=10).fill = Styles.gray_fill  # 浮动盈亏
        ws.cell(row=row_idx, column=11).fill = Styles.gray_fill  # 盈亏比例
    
    # 调整列宽
    column_widths = [12, 15, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 10, 10, 10, 10, 20]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    # 添加数据验证（所属板块）
    from openpyxl.worksheet.datavalidation import DataValidation
    dv = DataValidation(type="list", formula1='"上游油气，LNG,油服，煤炭化工，特高压，新能源，黄金，军工，政策方向"')
    dv.error = '请选择有效的板块名称'
    dv.errorTitle = '无效的板块'
    ws.add_data_validation(dv)
    dv.add(f'C2:C{len(STOCKS_DATA)+1}')
    
    # 添加数据验证（技术信号）
    dv2 = DataValidation(type="list", formula1='"多头，空头，震荡，观望"')
    ws.add_data_validation(dv2)
    dv2.add(f'R2:R{len(STOCKS_DATA)+1}')
    
    # 添加数据验证（基本面评级）
    dv3 = DataValidation(type="list", formula1='"优秀，良好，一般，警惕"')
    ws.add_data_validation(dv3)
    dv3.add(f'S2:S{len(STOCKS_DATA)+1}')
    
    print("✅ 工作表 2：持仓明细 - 创建完成")


def create_dashboard_sheet(wb):
    """创建工作表 1：组合概览"""
    ws = wb.create_sheet('01_组合概览', 0)
    
    # 标题
    ws.merge_cells('A1:G1')
    title = ws.cell(row=1, column=1, value='📊 投资组合跟踪仪表盘')
    title.font = Font(name='微软雅黑', size=18, bold=True, color='FFFFFF')
    title.fill = Styles.header_fill
    title.alignment = Styles.center_align
    
    # 核心指标
    metrics = [
        ['指标名称', '数值', '说明'],
        ['组合总市值', f'¥{INITIAL_CAPITAL:,}', '当前持仓总市值'],
        ['当日盈亏', '¥0', '当日浮动盈亏'],
        ['累计收益率', '0.00%', '相对于初始本金'],
        ['年化收益率', '0.00%', '年化计算'],
        ['最大回撤', '0.00%', '历史最大回撤'],
        ['夏普比率', '0.00', '风险调整后收益'],
        ['持仓数量', f'{len(STOCKS_DATA)}只', '当前持仓标的数'],
        ['现金比例', '3.00%', '现金占总资产比例'],
    ]
    
    for row_idx, row_data in enumerate(metrics, 3):
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.border = Styles.thin_border
            if col_idx == 1:
                cell.alignment = Styles.left_align
                cell.fill = Styles.gray_fill
            elif col_idx == 2:
                cell.alignment = Styles.right_align
                cell.font = Font(name='微软雅黑', size=10, bold=True)
            else:
                cell.alignment = Styles.left_align
    
    # 板块配置标题
    ws.cell(row=14, column=1, value='📈 板块配置分布').font = Font(name='微软雅黑', size=12, bold=True)
    
    # 板块数据
    sectors = {}
    for stock in STOCKS_DATA:
        sector = stock['板块']
        if sector not in sectors:
            sectors[sector] = 0
        sectors[sector] += stock['目标仓位']
    
    ws.cell(row=15, column=1, value='板块').font = Styles.header_font
    ws.cell(row=15, column=2, value='目标市值').font = Styles.header_font
    ws.cell(row=15, column=3, value='占比').font = Styles.header_font
    
    total_target = sum(sectors.values())
    for row_idx, (sector, target) in enumerate(sectors.items(), 16):
        ws.cell(row=row_idx, column=1, value=sector)
        ws.cell(row=row_idx, column=2, value=f'¥{INITIAL_CAPITAL * target / 100:,.0f}')
        ws.cell(row=row_idx, column=3, value=f'{target/total_target*100:.1f}%')
    
    # 预警信号
    ws.cell(row=14, column=5, value='⚠️ 预警信号').font = Font(name='微软雅黑', size=12, bold=True)
    alerts = [
        ['预警类型', '触发条件', '当前状态'],
        ['止损预警', '股价<止损价', '正常'],
        ['目标价预警', '股价>目标价 95%', '正常'],
        ['权重偏离', '偏离>15%', '正常'],
        ['股息率达标', '股息率>3%', '部分达标'],
        ['估值分位', 'PE 分位<30%', '合理'],
    ]
    
    for row_idx, row_data in enumerate(alerts, 15):
        for col_idx, value in enumerate(row_data, 5):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.border = Styles.thin_border
            cell.alignment = Styles.center_align
    
    # 调整列宽
    for col in 'ABCDEFG':
        ws.column_dimensions[col].width = 15
    
    print("✅ 工作表 1：组合概览 - 创建完成")


def create_transactions_sheet(wb):
    """创建工作表 3：交易记录"""
    ws = wb.create_sheet('03_交易记录')
    
    headers = [
        '交易日期', '股票代码', '股票名称', '交易类型', '交易价格', '交易数量',
        '交易金额', '手续费', '净现金流', '交易后持仓', '交易后成本',
        '交易理由', '操作人', '备注'
    ]
    
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 示例数据
    sample_data = [
        [datetime.now().strftime('%Y-%m-%d'), '600938', '中国海油', '买入', 40.00, 1000, 40000, 100, -40100, 1000, 40.00, '回调至支撑位建仓', '用户', '首仓'],
    ]
    
    for row_idx, row_data in enumerate(sample_data, 2):
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.alignment = Styles.center_align
            cell.border = Styles.thin_border
    
    # 调整列宽
    column_widths = [12, 12, 15, 10, 12, 12, 12, 12, 12, 12, 12, 20, 10, 20]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    print("✅ 工作表 3：交易记录 - 创建完成")


def create_fundamentals_sheet(wb):
    """创建工作表 4：财务监控"""
    ws = wb.create_sheet('04_财务监控')
    
    headers = [
        '股票代码', '股票名称', '报告期', '营收 (亿)', '营收同比%', '净利润 (亿)',
        '净利润同比%', '毛利率%', 'ROE%', '负债率%', '经营现金流 (亿)',
        '数据来源', '更新日期'
    ]
    
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 示例数据（每只股票 2 个报告期）
    row_idx = 2
    for stock in STOCKS_DATA[:5]:  # 示例只写前 5 只
        for period in ['2026Q1', '2025Q4']:
            data = [
                stock['代码'], stock['名称'], period,
                0, 0, 0, 0, 0, 0, 0, 0,
                '待更新', datetime.now().strftime('%Y-%m-%d')
            ]
            for col_idx, value in enumerate(data, 1):
                cell = ws.cell(row=row_idx, column=col_idx, value=value)
                cell.font = Styles.normal_font
                cell.alignment = Styles.center_align
                cell.border = Styles.thin_border
            row_idx += 1
    
    # 调整列宽
    column_widths = [12, 15, 10, 12, 12, 12, 12, 10, 10, 10, 12, 12, 12]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    print("✅ 工作表 4：财务监控 - 创建完成")


def create_technical_sheet(wb):
    """创建工作表 5：技术信号"""
    ws = wb.create_sheet('05_技术信号')
    
    headers = [
        '股票代码', '股票名称', '更新日期', '收盘价', '5 日均线', '20 日均线',
        '60 日均线', 'MACD', 'RSI(14)', '成交量 (万手)', '技术形态',
        '关键支撑', '关键阻力', '信号建议'
    ]
    
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 示例数据
    for row_idx, stock in enumerate(STOCKS_DATA, 2):
        data = [
            stock['代码'], stock['名称'], datetime.now().strftime('%Y-%m-%d'),
            stock['当前价'], 0, 0, 0, 0, 50, 0, '震荡',
            stock['入场下限'], stock['入场上限'], '持有'
        ]
        for col_idx, value in enumerate(data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.alignment = Styles.center_align
            cell.border = Styles.thin_border
    
    # 调整列宽
    column_widths = [12, 15, 12, 10, 10, 10, 10, 10, 10, 12, 10, 10, 10, 10]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    print("✅ 工作表 5：技术信号 - 创建完成")


def create_rebalance_sheet(wb):
    """创建工作表 6：再平衡日志"""
    ws = wb.create_sheet('06_再平衡日志')
    
    headers = [
        '检视日期', '触发原因', '调整标的', '调整前仓位%', '目标仓位%',
        '调整后仓位%', '操作动作', '交易金额', '调整后组合收益', '备注'
    ]
    
    for col_idx, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_idx, value=header)
        cell.font = Styles.header_font
        cell.fill = Styles.header_fill
        cell.alignment = Styles.center_align
        cell.border = Styles.thin_border
    
    # 示例数据
    sample_data = [
        [datetime.now().strftime('%Y-%m-%d'), '月度检视', '中国海油', 13.5, 10, 10.2, '减仓 3.3%', -32000, '8.35%', '油价冲高止盈部分'],
        [datetime.now().strftime('%Y-%m-%d'), '月度检视', '阳光电源', 6.8, 8, 7.9, '加仓 1.1%', 11000, '8.35%', '回调至支撑位加仓'],
    ]
    
    for row_idx, row_data in enumerate(sample_data, 2):
        for col_idx, value in enumerate(row_data, 1):
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Styles.normal_font
            cell.alignment = Styles.center_align
            cell.border = Styles.thin_border
    
    # 调整列宽
    column_widths = [12, 15, 15, 12, 12, 12, 12, 12, 12, 20]
    for i, width in enumerate(column_widths, 1):
        ws.column_dimensions[get_column_letter(i)].width = width
    
    print("✅ 工作表 6：再平衡日志 - 创建完成")


def add_conditional_formatting(wb):
    """添加条件格式（盈亏颜色）"""
    ws = wb['02_持仓明细']
    
    # 注意：openpyxl 的条件格式功能有限，这里用填充色模拟
    # 实际使用时建议在 Excel 中手动设置条件格式规则
    
    print("⚠️ 条件格式提示：建议在 Excel 中手动设置以下规则")
    print("   - J 列（浮动盈亏）：正值绿色，负值红色")
    print("   - K 列（盈亏比例）：>20% 深绿，10-20% 浅绿，-10-10% 黄，<-10% 红")
    print("   - U 列（操作建议）：加仓蓝色，持有灰色，止盈橙色，止损红色")


def add_formulas(wb):
    """添加 Excel 公式（持仓明细表）"""
    ws = wb['02_持仓明细']
    
    # 注意：openpyxl 写入公式后需要在 Excel 中计算
    # 这里添加公式示例
    
    # H 列：持仓市值 = E 列*G 列
    for row in range(2, len(STOCKS_DATA) + 2):
        ws.cell(row=row, column=8).value = f'=E{row}*G{row}'
    
    # I 列：实际仓位% = H 列/组合总市值
    for row in range(2, len(STOCKS_DATA) + 2):
        ws.cell(row=row, column=9).value = f'=H{row}/\'01 组合概览\'!$B$2'
    
    # J 列：浮动盈亏 = (G 列-F 列)*E 列
    for row in range(2, len(STOCKS_DATA) + 2):
        ws.cell(row=row, column=10).value = f'=(G{row}-F{row})*E{row}'
    
    # K 列：盈亏比例% = (G 列-F 列)/F 列
    for row in range(2, len(STOCKS_DATA) + 2):
        ws.cell(row=row, column=11).value = f'=(G{row}-F{row})/F{row}'
    
    print("✅ Excel 公式 - 添加完成（需在 Excel 中启用自动计算）")


# ==================== 主函数 ====================

def main():
    """主函数：生成 Excel 文件"""
    print("=" * 60)
    print("🚀 投资组合跟踪模板自动生成器")
    print("=" * 60)
    print(f"📅 生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"💰 初始本金：¥{INITIAL_CAPITAL:,}")
    print(f"📊 标的数量：{len(STOCKS_DATA)}只")
    print(f"📁 输出文件：{FILENAME}")
    print("=" * 60)
    
    # 创建工作簿
    wb = Workbook()
    
    # 删除默认 sheet
    default_sheet = wb.active
    wb.remove(default_sheet)
    
    # 创建所有工作表
    print("\n📋 开始创建工作表...")
    create_parameters_sheet(wb)      # 07_参数设置
    create_positions_sheet(wb)       # 02_持仓明细
    create_dashboard_sheet(wb)       # 01_组合概览
    create_transactions_sheet(wb)    # 03_交易记录
    create_fundamentals_sheet(wb)    # 04_财务监控
    create_technical_sheet(wb)       # 05_技术信号
    create_rebalance_sheet(wb)       # 06_再平衡日志
    
    # 添加公式和条件格式
    print("\n🔧 添加公式和格式...")
    add_formulas(wb)
    add_conditional_formatting(wb)
    
    # 调整工作表顺序
    wb.move_sheet('01_组合概览', offset=-6)  # 移到第一个
    
    # 保存文件
    print(f"\n💾 保存文件：{FILENAME}...")
    wb.save(FILENAME)
    
    print("\n" + "=" * 60)
    print("✅ Excel 模板生成成功！")
    print("=" * 60)
    print("\n📂 文件位置：当前目录下")
    print(f"📄 文件名：{FILENAME}")
    print("\n📋 包含工作表：")
    print("   1️⃣  01_组合概览（Dashboard）")
    print("   2️⃣  02_持仓明细（Positions）")
    print("   3️⃣  03_交易记录（Transactions）")
    print("   4️⃣  04_财务监控（Fundamentals）")
    print("   5️⃣  05_技术信号（Technical）")
    print("   6️⃣  06_再平衡日志（Rebalance）")
    print("   7️⃣  07_参数设置（Settings）")
    print("\n💡 使用提示：")
    print("   1. 打开 Excel 文件后启用'自动计算'")
    print("   2. 在'02_持仓明细'中更新持仓数量和成本")
    print("   3. 在'01_组合概览'中查看实时仪表盘")
    print("   4. 每月末执行再平衡并记录到'06_再平衡日志'")
    print("=" * 60)
    
    return FILENAME


# ==================== 运行 ====================

if __name__ == '__main__':
    try:
        filename = main()
        print(f"\n🎉 生成完成！文件：{filename}")
    except Exception as e:
        print(f"\n❌ 生成失败：{str(e)}")
        import traceback
        traceback.print_exc()

🚀 投资组合跟踪模板自动生成器
📅 生成时间：2026-03-20 10:31:27
💰 初始本金：¥100,000
📊 标的数量：18只
📁 输出文件：投资组合跟踪模板_20260320_103127.xlsx

📋 开始创建工作表...
✅ 工作表 7：参数设置 - 创建完成
✅ 工作表 2：持仓明细 - 创建完成
✅ 工作表 1：组合概览 - 创建完成
✅ 工作表 3：交易记录 - 创建完成
✅ 工作表 4：财务监控 - 创建完成
✅ 工作表 5：技术信号 - 创建完成
✅ 工作表 6：再平衡日志 - 创建完成

🔧 添加公式和格式...
✅ Excel 公式 - 添加完成（需在 Excel 中启用自动计算）
⚠️ 条件格式提示：建议在 Excel 中手动设置以下规则
   - J 列（浮动盈亏）：正值绿色，负值红色
   - K 列（盈亏比例）：>20% 深绿，10-20% 浅绿，-10-10% 黄，<-10% 红
   - U 列（操作建议）：加仓蓝色，持有灰色，止盈橙色，止损红色

💾 保存文件：投资组合跟踪模板_20260320_103127.xlsx...

✅ Excel 模板生成成功！

📂 文件位置：当前目录下
📄 文件名：投资组合跟踪模板_20260320_103127.xlsx

📋 包含工作表：
   1️⃣  01_组合概览（Dashboard）
   2️⃣  02_持仓明细（Positions）
   3️⃣  03_交易记录（Transactions）
   4️⃣  04_财务监控（Fundamentals）
   5️⃣  05_技术信号（Technical）
   6️⃣  06_再平衡日志（Rebalance）
   7️⃣  07_参数设置（Settings）

💡 使用提示：
   1. 打开 Excel 文件后启用'自动计算'
   2. 在'02_持仓明细'中更新持仓数量和成本
   3. 在'01_组合概览'中查看实时仪表盘
   4. 每月末执行再平衡并记录到'06_再平衡日志'

🎉 生成完成！文件：投资组合跟踪模板_20260320_103127.xlsx
